
## Transform Races Data
1. Read bronze races table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (raceName → race_name, circuitId → circuit_id)
4. Rename columns to make them more meaningful (date → race_date)
5. Remove duplicate records
6. Transform values of column race_name to Title Case
7. Write the transformed data to silver races table

In [0]:
# 1. Get batch ID
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/03_silver_helpers"

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_path = "s3://bizf1project123/Bronze/Kalashri_increment/races"

silver_path = "s3://bizf1project123/Silver/Kalashri_increment/races"

## Step 1 - Read bronze races table

In [0]:
# 5. Read Bronze Delta data for the selected batch
races_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
    .filter(F.col("batch_id") == v_batch_id)
)

display(races_df)

## Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
races_selected_df = races_df.select(
    F.col("season").alias("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

## Step 3 & 4 - Standardise Column Names
-    Standardise column names using snake_case (circuitId → circuit_id, raceName → race_name)
-    Rename columns to make them more meaningful (date → race_date)

In [0]:
races_renamed_df = (
    races_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "raceName": "race_name",
            "date": "race_date"
        })
)

display(races_renamed_df)

## Step 5 - Remove duplicate records

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season", "round"])
display(races_distinct_df)


## Step 6 - Transform values of column race_name to Title Case

In [0]:
races_final_df = (
    races_distinct_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)

display(races_final_df)

## Step 7 - Write the transformed data to silver races table


In [0]:

# Write transformed data to Silver
write_to_silver(
    input_df=races_final_df,
    target_path=silver_path,
    merge_condition="t.season = s.season AND t.round = s.round",
    columns_to_update=[
        "season",
        "round",
        "race_name",
        "circuit_id",
        "race_date",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)


In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Silver/Kalashri_increment/races",
#     True
# )